In [28]:
import pandas as pd 
import numpy as np
#import data
#lake elevation and precipitation (monthly)
elev_prec_df = pd.read_csv('compiled_lake_data.csv')
#infow acft
inflow_acft_df = pd.read_csv('gsl_inflow_monthly_acft.csv')

#Bigalke et al. (2025) original equation:
#V_t = V_(t-1) + P_t × A_(t-1) + R_t − E_t × A_(t-1)
#https://agupubs.onlinelibrary.wiley.com/doi/epdf/10.1029/2024GL112154
#Rearranged and simplified equation:
#E_t = V_(t-1) + R_t + P_t − V_t

In [29]:
#to calculate how much water is in the lake (not just height/elevation), need to convert elevation to volume acre-feet
#import stage-volume curve for GSL: 
#tells us how much water is in the lake at an elevetion of 4197ft (~above sea level, because the lake is wider/narrower at different places)
#https://www.sciencebase.gov/catalog/item/6467b42fd34ec11ae4a8afb1
stage_vol_df = pd.read_csv('GSL_ElevAreaVolume_total.csv')
print(stage_vol_df.head())
print(stage_vol_df.columns)


   elev_ft_NAVD88  elev_ft_NGVD29  area_ft2  area_m2  area_km2  area_mi2  \
0         4170.00         4166.52       0.0        0       0.0       0.0   
1         4170.01         4166.53       0.0        0       0.0       0.0   
2         4170.02         4166.54       0.0        0       0.0       0.0   
3         4170.03         4166.55       0.0        0       0.0       0.0   
4         4170.04         4166.56       0.0        0       0.0       0.0   

   area_acre  volume_ft3  volume_m3  volume_km3  volume_mi3  volume_acreft  
0        0.0         0.0        0.0         0.0         0.0            0.0  
1        0.0         0.0        0.0         0.0         0.0            0.0  
2        0.0         0.0        0.0         0.0         0.0            0.0  
3        0.0         0.0        0.0         0.0         0.0            0.0  
4        0.0         0.0        0.0         0.0         0.0            0.0  
Index(['elev_ft_NAVD88', 'elev_ft_NGVD29', 'area_ft2', 'area_m2', 'area_km2',
   

In [30]:
#reload original data to start clean
elev_prec_df = pd.read_csv('compiled_lake_data.csv')
#round elevation to 2 decimal places to match the stage-volume curve
elev_prec_df['elevation_ft'] = elev_prec_df['elevation_ft'].round(2)
#match each lake elevation in elev_prec_df to its corresponding volume and surface area in stage_vol_df
elev_prec_df = elev_prec_df.merge(
    #grab these columns from the stage-volume table
    stage_vol_df[['elev_ft_NGVD29', 'volume_acreft', 'area_acre']],
    #match rows where elevation_ft == elev_ft_NGVD29 
    #(NGVD29 = at mean sea level taken in 1929)
    left_on = 'elevation_ft',
    right_on = 'elev_ft_NGVD29',
    #keep all rows from elev_prec_df
    how = 'left'
)
elev_prec_df.head()


,Unnamed: 0,date,year,month,elevation_ft,monthly_precip_inches,snow_water_equiv_in,elev_ft_NGVD29,volume_acreft,area_acre
0,0,1950-01-01,1950,1,4197.15,1.80,NaN,4197.15,13013628.81,864498.36
1,1,1950-02-01,1950,2,4197.40,1.19,NaN,4197.40,13231625.08,879251.73
2,2,1950-03-01,1950,3,4197.90,0.98,NaN,4197.90,13677742.99,905185.46
3,3,1950-04-01,1950,4,4198.15,1.29,NaN,4198.15,13905660.50,918138.01
4,4,1950-05-01,1950,5,4198.35,2.18,NaN,4198.35,14090506.69,928507.42


In [31]:
#shift surface area and volume one row to get prior month values, to match Bigalke et al. (2025)
#use prior month surface area (A_(t-1))
elev_prec_df['area_acre_prior'] = elev_prec_df['area_acre'].shift(1)
#calculate V_(t-1) by grabing volume from the previous row 
elev_prec_df['prior_month_vol_acft'] = elev_prec_df['volume_acreft'].shift(1)
elev_prec_df.head()


,Unnamed: 0,date,year,month,elevation_ft,monthly_precip_inches,snow_water_equiv_in,elev_ft_NGVD29,volume_acreft,area_acre,area_acre_prior,prior_month_vol_acft
0,0,1950-01-01,1950,1,4197.15,1.80,NaN,4197.15,13013628.81,864498.36,NaN,NaN
1,1,1950-02-01,1950,2,4197.40,1.19,NaN,4197.40,13231625.08,879251.73,864498.36,13013628.81
2,2,1950-03-01,1950,3,4197.90,0.98,NaN,4197.90,13677742.99,905185.46,879251.73,13231625.08
3,3,1950-04-01,1950,4,4198.15,1.29,NaN,4198.15,13905660.50,918138.01,905185.46,13677742.99
4,4,1950-05-01,1950,5,4198.35,2.18,NaN,4198.35,14090506.69,928507.42,918138.01,13905660.50


In [32]:
#convert precipitation from inches to acre-feet 
#12 inches in a ft 
elev_prec_df['precip_acreft'] = (elev_prec_df['monthly_precip_inches'] / 12) * elev_prec_df['area_acre_prior']
elev_prec_df.head()


,Unnamed: 0,date,year,month,elevation_ft,monthly_precip_inches,snow_water_equiv_in,elev_ft_NGVD29,volume_acreft,area_acre,area_acre_prior,prior_month_vol_acft,precip_acreft
0,0,1950-01-01,1950,1,4197.15,1.80,NaN,4197.15,13013628.81,864498.36,NaN,NaN,NaN
1,1,1950-02-01,1950,2,4197.40,1.19,NaN,4197.40,13231625.08,879251.73,864498.36,13013628.81,85729.420700
2,2,1950-03-01,1950,3,4197.90,0.98,NaN,4197.90,13677742.99,905185.46,879251.73,13231625.08,71805.557950
3,3,1950-04-01,1950,4,4198.15,1.29,NaN,4198.15,13905660.50,918138.01,905185.46,13677742.99,97307.436950
4,4,1950-05-01,1950,5,4198.35,2.18,NaN,4198.35,14090506.69,928507.42,918138.01,13905660.50,166795.071817


In [33]:
#datetime: convert date columns so they match when merged 
elev_prec_df['date'] = pd.to_datetime(elev_prec_df['date'])
inflow_acft_df['date'] = pd.to_datetime(inflow_acft_df['date'])

#merge inflow data with main dataframe (elevation and precipitation)
elev_prec_df = elev_prec_df.merge(
    #grab only the date and total inflow column
    inflow_acft_df[['date', 'TOTAL_INFLOW_cfs_acft']],
    #match rows wherre the date is the same
    on = 'date',
    #keep rows from elev_prec_df if nothing matches
    how = 'left'
)
elev_prec_df.head()

,Unnamed: 0,date,year,month,elevation_ft,monthly_precip_inches,snow_water_equiv_in,elev_ft_NGVD29,volume_acreft,area_acre,area_acre_prior,prior_month_vol_acft,precip_acreft,TOTAL_INFLOW_cfs_acft
0,0,1950-01-01,1950,1,4197.15,1.80,NaN,4197.15,13013628.81,864498.36,NaN,NaN,NaN,311399.5825
1,1,1950-02-01,1950,2,4197.40,1.19,NaN,4197.40,13231625.08,879251.73,864498.36,13013628.81,85729.420700,330234.8985
2,2,1950-03-01,1950,3,4197.90,0.98,NaN,4197.90,13677742.99,905185.46,879251.73,13231625.08,71805.557950,378388.3280
3,3,1950-04-01,1950,4,4198.15,1.29,NaN,4198.15,13905660.50,918138.01,905185.46,13677742.99,97307.436950,607916.9645
4,4,1950-05-01,1950,5,4198.35,2.18,NaN,4198.35,14090506.69,928507.42,918138.01,13905660.50,166795.071817,811463.7345


In [34]:
#calculate evaporation using the mass-balance equation
#E_t = V_(t-1) + R_t + P_t - V_t
elev_prec_df['evaporation_acreft'] = (
    elev_prec_df['prior_month_vol_acft']      # V_(t-1)
    + elev_prec_df['TOTAL_INFLOW_cfs_acft']  # R_t
    + elev_prec_df['precip_acreft']          # P_t
    - elev_prec_df['volume_acreft']          # V_t
)

print(elev_prec_df[['date', 'prior_month_vol_acft', 'TOTAL_INFLOW_cfs_acft', 
                     'precip_acreft', 'volume_acreft', 
                     'evaporation_acreft']].head())


        date  prior_month_vol_acft  TOTAL_INFLOW_cfs_acft  precip_acreft  \
0 1950-01-01                   NaN            311399.5825            NaN   
1 1950-02-01           13013628.81            330234.8985   85729.420700   
2 1950-03-01           13231625.08            378388.3280   71805.557950   
3 1950-04-01           13677742.99            607916.9645   97307.436950   
4 1950-05-01           13905660.50            811463.7345  166795.071817   

   volume_acreft  evaporation_acreft  
0    13013628.81                 NaN  
1    13231625.08       197968.049200  
2    13677742.99         4075.975950  
3    13905660.50       477306.891450  
4    14090506.69       793412.616317  


In [37]:
#drop the first row since V_(t-1) is NaN for January 1950, there is no previous year data 
evaporation_data = elev_prec_df.dropna(subset = ['evaporation_acreft'])
#save to CSV
evaporation_data.to_csv('evaporation_data.csv', index = False)

print(evaporation_data.head())
print(f'Total rows: {len(evaporation_data)}')

   Unnamed: 0       date  year  month  elevation_ft  monthly_precip_inches  \
1           1 1950-02-01  1950      2       4197.40                   1.19   
2           2 1950-03-01  1950      3       4197.90                   0.98   
3           3 1950-04-01  1950      4       4198.15                   1.29   
4           4 1950-05-01  1950      5       4198.35                   2.18   
5           5 1950-06-01  1950      6       4198.80                   0.16   

   snow_water_equiv_in  elev_ft_NGVD29  volume_acreft  area_acre  \
1                  NaN         4197.40    13231625.08  879251.73   
2                  NaN         4197.90    13677742.99  905185.46   
3                  NaN         4198.15    13905660.50  918138.01   
4                  NaN         4198.35    14090506.69  928507.42   
5                  NaN         4198.80    14513327.20  951733.99   

   area_acre_prior  prior_month_vol_acft  precip_acreft  \
1        864498.36           13013628.81   85729.420700   
2   